# Analysis of Co-Design Experiments

This notebook loads the JSON result files produced by `run_experiment.py` and visualizes the key findings of the paper. It allows for the reproduction of the main figures and tables.

## 1. Load Experiment Results

First, we scan the `outputs/` directory (which is the default output for Hydra) to find all `*_metrics.json` files. We load them into a pandas DataFrame for easy analysis.

In [ ]:
import json
import glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.float_format', lambda x: f'{x:.3f}')
sns.set_theme(style="whitegrid")

output_dir = Path("../outputs") # Assuming notebooks are run from the `notebooks` dir
all_files = glob.glob(str(output_dir / "**" / "*_metrics.json"), recursive=True)
records = []
for f in all_files:
    with open(f) as fp:
        data = json.load(fp)
    # Extract the method name from the file path
    data["method"] = Path(f).stem.replace("_metrics", "")
    records.append(data)

if records:
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} experiment results.")
    display(df.head())
else:
    print("No result files found in the '../outputs' directory.")

## 2. Summary Table (Reproducing Table 1)

This section generates a summary table that compares the performance of all co-design methods. The key metrics are:
- **Task Performance**: Perplexity or Accuracy (lower is better for perplexity, higher for accuracy).
- **Latency**: The inference time in milliseconds (lower is better).
- **Modularity**: A measure of how well-clustered the memory layout is (higher is better).
- **L2 Cache Hit Rate**: The percentage of memory requests served by the L2 cache (higher is better).

In [ ]:
if records:
    # Dynamically select the task performance metric available in the dataframe
    perf_metric = "perplexity" if "perplexity" in df.columns else "accuracy"
    
    summary_cols = [
        "method",
        perf_metric,
        "latency_ms",
        "modularity",
        "l2_cache_hit_rate_pct",
    ]
    
    # Ensure all required columns exist
    summary_cols = [col for col in summary_cols if col in df.columns]
    
    summary = df[summary_cols].sort_values("latency_ms").set_index("method")
    display(summary)
else:
    print("Populate `../outputs/` with experiment results to view the summary table.")

## 3. Causal Chain Analysis (Reproducing Figure 3)

This plot visualizes the core hypothesis of the paper: that the iterative co-design process creates a causal chain where improvements in one domain enable improvements in the other. We expect to see latency decrease as modularity and cache hit rate increase over the iterations.

In [ ]:
if records and "iteration" in df.columns:
    iter_df = df[df['method'] == 'iterative'].sort_values("iteration")
    
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Plot Latency on the primary y-axis
    color = 'tab:blue'
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Latency (ms)", color=color)
    ax1.plot(iter_df["iteration"], iter_df["latency_ms"], color=color, marker='o', label="Latency (ms)")
    ax1.tick_params(axis='y', labelcolor=color)

    # Create a secondary y-axis for Modularity and Cache Hit Rate
    ax2 = ax1.twinx()
    color = 'tab:green'
    ax2.set_ylabel("Score / %", color=color)
    ax2.plot(iter_df["iteration"], iter_df["modularity"], color='tab:orange', marker='s', linestyle='--', label="Modularity")
    ax2.plot(iter_df["iteration"], iter_df["l2_cache_hit_rate_pct"], color='tab:green', marker='^', linestyle=':', label="L2 Cache Hit Rate (%)")
    ax2.tick_params(axis='y', labelcolor=color)

    fig.tight_layout() # Adjust layout to prevent labels from overlapping
    fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.9))
    plt.title("Performance Metrics vs. Iteration")
    plt.show()
else:
    print("No iteration data available. Run the 'iterative' method to generate this plot.")

## 4. Pareto Frontier Analysis (Reproducing Figure 4)

The Pareto frontier plot helps visualize the trade-off between task performance (e.g., perplexity) and hardware efficiency (latency). An ideal method pushes the frontier towards the bottom-left (lower latency and lower perplexity). We expect the `iterative` method to be on or near the optimal frontier.

In [ ]:
if records:
    perf_metric = "perplexity" if "perplexity" in df.columns else "accuracy"

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=df, x=perf_metric, y="latency_ms", hue="method", s=150, style="method")
    
    plt.title(f"Pareto Frontier: Latency vs. {perf_metric.title()}")
    plt.xlabel(f"{perf_metric.title()} (Lower is Better)")
    plt.ylabel("Latency (ms) (Lower is Better)")
    plt.legend(title="Method", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True)
    plt.show()
else:
    print("No result files found to plot the Pareto frontier.")